# Lexicon for ChatGPT (no API key needed)
Use P1's or P2's personal style database with **ChatGPT the app** instead of the API.

| Mode | What you do | Cost |
|---|---|---|
| **1. Paste mode** | Type the message + what you mean here, copy the printed prompt into chatgpt.com | free |
| **2. Check + learn** | Paste ChatGPT's reply back, run local checks, keep or edit it, and it's logged for the database | free |
| **3. Custom GPT / Project** | Export instructions + a knowledge file once, upload them to a Custom GPT or ChatGPT Project, then just chat | free |
| 4. API (optional) | Same as the main notebook, only if `OPENAI_API_KEY` is set | ~$0.0001/reply |

Retrieval and checks run locally on CPU (MiniLM + StyleDistance). Kept replies are saved as `gpt_confirmed` / `gpt_edited`, the same origins the main notebook uses for condition D.

In [1]:
# pip install sentence-transformers pandas numpy
import sys, os, warnings
os.environ.update(HF_HUB_DISABLE_PROGRESS_BARS="1", TRANSFORMERS_VERBOSITY="error", TOKENIZERS_PARALLELISM="false")
warnings.filterwarnings("ignore")
sys.path.insert(0, "code")
from lexicon_chatgpt import load_store, style_card, paste_prompt, check_reply, log_feedback, feedback_rows, export_custom_gpt, LABELS

PERSON = "P1"          # "P1" or "P2"
EXCLUDE_TEST = True    # True keeps the frozen test scenes out (clean evaluation). False = use every reply (everyday use).
store = load_store(PERSON, exclude_test=EXCLUDE_TEST, extra_rows=feedback_rows(PERSON))
print(PERSON, "database rows:", len(store.df))
print(style_card(PERSON, store))

This model was loaded in torch.bfloat16 on CPU. Depending on your CPU, inference may be significantly slower than with torch.float32. Consider comparing inference speeds by loading with `model_kwargs={"dtype": torch.float32}`.


P1 database rows: 49
length: median 3 words (range 1-8)
casing: capitalised first letter 100%; no final punctuation 94%
punctuation: '!' 2%, '...' 0%, space before '?' 4%
contractions: 31% of replies
elongated words (yesss): 4%
emoji: none
frequent words: "yeah", "sure", "idk"


## 1. Paste mode
Fill in the message and what you want to say. Copy everything between the lines into a new ChatGPT chat.

In [2]:
INCOMING = "TM: can you also cover the demo tomorrow? I'm swamped"
MEANING  = "I can't, I already have too much on my plate this week"     # what you want to say (content)
CONTEXT  = "teammate, class project, deadline week"                     # optional
INTENT   = "Push Back on Request"                                        # optional, one of LABELS or None
OPTIONS  = 1                                                            # 3 = ask ChatGPT for 3 candidates

prompt, refs = paste_prompt(PERSON, store, INCOMING, MEANING, context=CONTEXT, label=INTENT, n_options=OPTIONS)
print("-" * 70); print(prompt); print("-" * 70)

----------------------------------------------------------------------
INSTRUCTIONS
You write a single chat reply as P1, a real person.
CONTENT comes only from INTENDED MEANING. Do not add facts, plans, names, or reasons that are not in it.
STYLE comes from the STYLE CARD and the STYLE REFERENCES.
STYLE REFERENCES are past replies to OTHER messages. Use them only for tone, length, casing, punctuation, word choice and how this person softens or sharpens things. Never reuse their wording, topics, or facts. Do not copy any phrase of 4 or more words from them.
Match the card's length and casing exactly, even if that looks blunt or unpolished.
Output only the reply text. No quotes, no explanation.

STYLE CARD
length: median 3 words (range 1-8)
casing: capitalised first letter 100%; no final punctuation 94%
punctuation: '!' 2%, '...' 0%, space before '?' 4%
contractions: 31% of replies
elongated words (yesss): 4%
emoji: none
frequent words: "yeah", "sure", "idk"

STYLE REFERENCES (style only

## 2. Check + learn
Paste ChatGPT's reply into `GPT_REPLY`. If you'd change it, put your version in `FINAL_REPLY`; otherwise leave it the same. Set `KEEP = True` to save it to the database.

In [3]:
GPT_REPLY   = "Cant this week, already swamped"
FINAL_REPLY = GPT_REPLY          # or your edited version
KEEP = False                     # True = log it (gpt_confirmed if unchanged, gpt_edited if you edited)

print(check_reply(PERSON, store, GPT_REPLY, refs))
if KEEP:
    print(log_feedback(PERSON, INCOMING, MEANING, GPT_REPLY, FINAL_REPLY, context=CONTEXT, label=INTENT or ""))
    store = load_store(PERSON, exclude_test=EXCLUDE_TEST, extra_rows=feedback_rows(PERSON))   # rebuild with the new row

{'words': 5, 'typical_range': '1-5', 'length_ok': True, 'copied_4grams': [], 'stock_phrases': []}


## 3. Custom GPT / ChatGPT Project export
Writes two files per person into `chatgpt_export/`:
- `instructions_P1.md` goes in the GPT's **Instructions** box (or the Project's instructions). It's under the 8,000-character limit.
- `P1_examples.md` gets uploaded as a **Knowledge** file. ChatGPT's own file search then does the retrieval.

Then message it like this:
```
Message: can you also cover the demo tomorrow? I'm swamped
Meaning: I can't, too much on my plate this week
Context: teammate, deadline week
```
Re-export after logging new replies so the knowledge file grows. For the evaluation, keep `EXCLUDE_TEST=True` so test scenes never reach the GPT.

In [4]:
for p in ["P1", "P2"]:
    s = load_store(p, exclude_test=EXCLUDE_TEST, extra_rows=feedback_rows(p))
    ins, kb, n_chars, n_rows = export_custom_gpt(p, s)
    print(f"{p}: {ins.name} ({n_chars} chars), {kb.name} ({n_rows} examples)")
print(open(f"chatgpt_export/instructions_{PERSON}.md").read())

P1: instructions_P1.md (2770 chars), P1_examples.md (49 examples)


P2: instructions_P2.md (2857 chars), P2_examples.md (50 examples)
You are P1's reply writer. You draft ONE chat reply that sounds exactly like P1 texting.

HOW THE USER TALKS TO YOU
The user sends:
  Message: <what the other person wrote>
  Meaning: <what P1 wants to say back>      (required; this is the ONLY source of content)
  Context: <optional: who it is, situation>
  Intent:  <optional, one of: Acknowledge, Full Agree, Partial Agree + Modification, Full Disagree, Refuse / Decline, Suggest / Propose, Correct / Restate, Reverify / Confirm, 5W1H Question, Elaborate / Expand, Summarize / Condense, Soften / Hedge, Direct / Blunt, Express Emotion, Complain / Vent, Reject Idea with Alternative, Push Back on Request, Express Disappointment, Set a Boundary, Sarcasm / Teasing>
If Meaning is missing, ask for it in one short line. Do not guess it.

STEPS (do them silently)
1. Search the knowledge file P1_examples.md for 3-5 past replies with the same intent and a similar situation.
   They s

## 4. Optional: API mode
Skipped automatically if no `OPENAI_API_KEY` is set.

In [5]:
if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    from lexicon_gen_judge import generate_reply
    r = generate_reply(OpenAI(), PERSON, MEANING, INCOMING.split(":",1)[-1].strip(), context=CONTEXT,
                       style_card=style_card(PERSON, store), refs=refs)
    print(r.reply, check_reply(PERSON, store, r.reply, refs))
else:
    print("No OPENAI_API_KEY set, so API mode is skipped. Modes 1-3 don't need it.")

No OPENAI_API_KEY set, so API mode is skipped. Modes 1-3 don't need it.
